# Lab 4: Develop a Multi-Agent System

In this lab, we build a modern, production-ready multi-agent system using the latest Azure AI Python SDKs and best practices.
- Each agent is created as a connected agent using Microsoft Foundry Agent Service.
- Orchestration is performed using direct agent-to-agent calls, not just a group chat or plugin pattern.
- Uses official Microsoft documentation patterns for agent creation, tool/resource registration, and message passing.


### Part 1: Create the Search, Report, and Validation Agents

#### Step 1: Load packages

In [ ]:
import os
import json
import time
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, AzureAISearchTool, AzureAISearchToolResource, AISearchIndexResource

# Load environment variables
load_dotenv()


#### Step 2: Connect to your Microsoft Foundry Project

Use a token credential for project and agent operations. The code tries `AzureCliCredential` first and falls back to `DefaultAzureCredential` if needed.

In [ ]:
# Connecting to our Microsoft Foundry project
project = AIProjectClient(
    endpoint=os.getenv("AIPROJECT_ENDPOINT"),
    credential=DefaultAzureCredential()
)

#### Step 3: Connect to Azure AI Search

In [3]:
# First enter the name of your search index

index_name="health-plan"
print(index_name)

In [ ]:
# Find Azure Cognitive Search connection
conn_id = None
for conn in project.connections.list():
    if getattr(conn, "type", None) == "CognitiveSearch":
        conn_id = conn.id
        break
if not conn_id:
    raise ValueError("No Azure Cognitive Search connection found in this project.")

# Build the Azure AI Search tool using azure.ai.projects.models classes
ai_search_tool = AzureAISearchTool(
    azure_ai_search=AzureAISearchToolResource(
        indexes=[
            AISearchIndexResource(
                project_connection_id=conn_id,
                index_name=index_name  # Be sure to set your index name above
            )
        ]
    )
)
print(f"Configured AI Search tool for index: {index_name}")

#### Step 4: Create the Search Agent
To create the Search Agent, we use the Microsoft Foundry Agent Service SDK to define a dedicated agent that specializes in searching our Azure AI Search index for health plan documents.

In [ ]:
# Search Agent - uses Azure AI Search tool to retrieve health plan information
agent_name = "search-agent"
agent_definition = PromptAgentDefinition(
    model=os.getenv("CHAT_MODEL"),
    instructions="You are a helpful agent that is an expert at searching health plan documents. Use the Azure AI Search tool to retrieve relevant information, then summarize what you find.",
    tools=[ai_search_tool],
)
search_agent = project.agents.create_version(
    agent_name=agent_name,
    definition=agent_definition,
)
print(f"Created search agent, ID: {search_agent.id}")

#### Step 5: Create the Report Agent
Similarly, to create the Report Agent, we use the Micorosft Foundry Agent Service SDK to define an agent dedicated to generating detailed reports about health plans. This agent is configured with a specialized system prompt and can be easily orchestrated alongside other agents in the workflow.

In [ ]:
# Search Agent - uses Azure AI Search tool to retrieve health plan information
agent_name = "search-agent"
agent_definition = PromptAgentDefinition(
    model=os.getenv("CHAT_MODEL"),
    instructions="You are a helpful agent that is an expert at searching health plan documents. Use the Azure AI Search tool to retrieve relevant information, then summarize what you find.",
    tools=[ai_search_tool],
)
search_agent = project.agents.create_version(
    agent_name=agent_name,
    definition=agent_definition,
)
print(f"Created search agent, ID: {search_agent.id}")

#### Step 6: Create the Validation Agent
To create the Validation Agent, we again use the Microsoft Foundry Service SDK to define an agent focused on validating that generated reports meet specific requirements. The Validation Agent is configured with instructions to check for required content (such as coverage exclusions) and to return a simple pass/fail result. This agent can be invoked programmatically as part of the multi-agent workflow, ensuring that all generated reports adhere to business rules before being delivered to the user.

In [ ]:
# Validation Agent
agent_definition = PromptAgentDefinition(
    model=os.getenv("CHAT_MODEL"),
    instructions="You are a helpful agent that validates reports. Return 'Pass' if the report meets requirements (must include coverage exclusions), otherwise return 'Fail'. Only return 'Pass' or 'Fail'."
)
validation_agent = project.agents.create_version(
    agent_name="validation-agent",
    definition=agent_definition,
)

Part 2: Orchestrate the Multi-Agent System

Now that you've created the Search, Report, and Validation agents, you can orchestrate them to generate a health plan report.

When you run the orchestration code cell, enter a health plan name at the input prompt. Example plans:

- Northwind Standard
- Northwind Health Plus

The orchestration code below:

- Defines helper functions to interact with the Search, Report, and Validation agents using the Azure AI Foundry SDK.
- Defines an orchestrate function that coordinates the multi-agent workflow:
- The Search Agent retrieves information about the specified health plan.
- The Report Agent generates a detailed report using the information returned by the Search Agent.
- The Validation Agent verifies that the report includes the required coverage exclusions and returns a validation result.
- If validation succeeds, the report is saved as a Markdown (.md) file; otherwise, a message is displayed indicating that the report did not meet the validation requirements.
- Provides an interactive input loop that allows you to enter health plan names and generate validated reports.

This workflow demonstrates how multiple AI agents can collaborate to retrieve information, generate content, validate the results, and produce a final report through an orchestrated sequence of interactions.

In [ ]:
def extract_last_agent_message(output_text):
    """Extract text from agent response."""
    return output_text if output_text else ""


def run_search_agent(user_content: str) -> str:
    """Run the search agent using the new Foundry SDK and return the response text."""
    openai = project.get_openai_client(agent_name="search-agent")
    conversation = openai.conversations.create()
    
    response = openai.responses.create(
        conversation=conversation.id,
        input=user_content,
    )
    
    return response.output_text or ""


def run_simple_agent(agent_name: str, user_content: str) -> str:
    """Run a no-tool agent (report or validation) using the new Foundry SDK and return the response text."""
    openai = project.get_openai_client(agent_name=agent_name)
    conversation = openai.conversations.create()
    
    response = openai.responses.create(
        conversation=conversation.id,
        input=user_content,
    )
    
    return response.output_text or ""


def orchestrate(plan_name: str):
    print(f"\n[1/3] Search agent retrieving info about '{plan_name}'...")
    plan_info = run_search_agent(f"Tell me about the {plan_name} plan.")

    print(f"[2/3] Report agent writing the report...")
    report_content = run_simple_agent(
        "report-agent",
        f"Write a detailed report about the {plan_name} plan. Include coverage exclusions. Here is the relevant information: {plan_info}"
    )

    print(f"[3/3] Validation agent checking the report...")
    validation_result = run_simple_agent(
        "validation-agent",
        f"Validate that the following report includes coverage exclusions. Here is the report: {report_content}"
    )

    if validation_result.strip().lower() == "pass":
        filename = f"{plan_name} Report.md"
        with open(filename, "w", encoding="utf-8") as f:
            f.write(report_content)
        print(f"Report validated and saved to: {filename}")
        return {"report_was_generated": True, "content": report_content}
    else:
        print(f"Validation result: {validation_result!r} - report did not meet requirements.")
        return {"report_was_generated": False, "content": "The report could not be generated as it did not meet the required validation standards."}


# Interactive loop - enter a health plan name when prompted
# Available plans: 'Northwind Standard' or 'Northwind Health Plus'
print("Welcome to the Health Plan Multi-Agent System!")
print("Available plans: 'Northwind Standard', 'Northwind Health Plus'")
while True:
    plan_name = input("Enter a health plan name (or 'exit' to quit): ").strip()
    if not plan_name or plan_name.lower() == "exit":
        break
    result = orchestrate(plan_name)
    print(json.dumps({"report_was_generated": result["report_was_generated"]}, indent=2))